In [1]:
import numpy as np
import warnings
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize
from scipy.stats import norm
warnings.filterwarnings('ignore')

In [2]:
data_in = np.load('initial_inputs.npy')
data_out = np.load('initial_outputs.npy')

X_init = np.array([
 [0.66579958, 0.12396913],
 [0.87779099, 0.7786275 ],
 [0.14269907, 0.34900513],
 [0.84527543, 0.71112027],
 [0.45464714, 0.29045518],
 [0.57771284, 0.77197318],
 [0.43816606, 0.68501826],
 [0.34174959, 0.02869772],
 [0.33864816, 0.21386725],
 [0.70263656, 0.9265642 ]]
)

y_init = np.array([
    0.53899612,
    0.42058624,
    -0.06562362,
    0.29399291,
    0.21496451,
    0.02310555,
    0.24461934,
    0.03874902,
    -0.01385762,
    0.61120522])

#week01's result
X_week2_add = np.array([1.000000, 0.000000])
y_week2_add = np.array([2.308810742126141e-248])

X_all = np.vstack([X_init, X_week2_add])
y_all = np.concatenate([y_init, y_week2_add])

X_init = X_all
y_init = y_all

print(X_init)
print(y_init)

[[0.66579958 0.12396913]
 [0.87779099 0.7786275 ]
 [0.14269907 0.34900513]
 [0.84527543 0.71112027]
 [0.45464714 0.29045518]
 [0.57771284 0.77197318]
 [0.43816606 0.68501826]
 [0.34174959 0.02869772]
 [0.33864816 0.21386725]
 [0.70263656 0.9265642 ]
 [1.         0.        ]]
[ 5.38996120e-001  4.20586240e-001 -6.56236200e-002  2.93992910e-001
  2.14964510e-001  2.31055500e-002  2.44619340e-001  3.87490200e-002
 -1.38576200e-002  6.11205220e-001  2.30881074e-248]


Week-01

In [3]:
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    sigma = sigma[0]
    mu = mu[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    from scipy.stats import norm
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative because we will minimize

# --- Optimize acquisition function ---
y_best = y_init.max()
bounds = [(0,1), (0,1)]

res = minimize(lambda x: expected_improvement(x, gp, y_best),
               x0=np.random.rand(2),
               bounds=bounds,
               method='L-BFGS-B')

x_next = res.x
print("Next point to evaluate:", x_next)



Next point to evaluate: [0.35357894 0.47454704]


Week-02

In [4]:
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-3, normalize_y=True)
gp.fit(X_init, y_init)

GaussianProcessRegressor(alpha=0.001, kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)

In [5]:
def expected_improvement(x, gp, y_best, xi=0.05):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    sigma = sigma[0]
    mu = mu[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    from scipy.stats import norm
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative because we will minimize

In [6]:
y_best = y_all.max()
bounds = [(0, 1), (0, 1)]

def propose_location(acquisition, gp, y_best, bounds, n_restarts=25):
    best_x = None
    best_acq = 1e10
    for i in range(n_restarts):
        x0 = np.random.rand(2)
        res = minimize(lambda x: acquisition(x, gp, y_best),
                       x0=x0,
                       bounds=bounds,
                       method='L-BFGS-B')
        if res.fun < best_acq:
            best_acq = res.fun
            best_x = res.x
    return best_x

x_next = propose_location(expected_improvement, gp, y_best, bounds)
print("Next point to evaluate:", x_next)



Next point to evaluate: [0.64868196 0.48446846]


Week-03

In [ ]:
X_init = np.array([
    [0.66579958, 0.12396913],
    [0.87779099, 0.7786275 ],
    [0.14269907, 0.34900513],
    [0.84527543, 0.71112027],
    [0.45464714, 0.29045518],
    [0.57771284, 0.77197318],
    [0.43816606, 0.68501826],
    [0.34174959, 0.02869772],
    [0.33864816, 0.21386725],
    [0.70263656, 0.9265642 ]
])

y_init = np.array([0.53899612, 0.42058624, -0.06562362, 0.29399291, 
                   0.21496451, 0.02310555, 0.24461934, 0.03874902, 
                   -0.01385762, 0.61120522])

#week02
X_new = np.array([
    [1.000000, 0.000000],
    [0.086648, 0.266342]
])
y_new = np.array([0.014684903068384703, 0.07375385295046949])

# Combine all data
X_all = np.vstack((X_init, X_new))
y_all = np.hstack((y_init, y_new))

# --- Fit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-3, normalize_y=True)
gp.fit(X_all, y_all)

# --- Expected Improvement ---
def expected_improvement(x, gp, y_best, xi=0.05):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu = mu[0]
    sigma = sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative because we will minimize

# --- Acquisition optimizer with multiple restarts ---
def propose_location(acquisition, gp, y_best, bounds, n_restarts=25):
    best_x = None
    best_acq = 1e10
    for _ in range(n_restarts):
        x0 = np.random.rand(2)
        res = minimize(lambda x: acquisition(x, gp, y_best),
                       x0=x0,
                       bounds=bounds,
                       method='L-BFGS-B')
        if res.fun < best_acq:
            best_acq = res.fun
            best_x = res.x
    return best_x

# --- Select next point ---
y_best = y_all.max()
bounds = [(0,1), (0,1)]
x_next = propose_location(expected_improvement, gp, y_best, bounds)

res_formatted = [f"{r:.6f}" for r in x_next]
result = "-".join(res_formatted)
print(result)



0.195799-0.810124
